# W03 · Grid encoders & the bottleneck (the crux)
# W03 · Grid 編碼器與瓶頸(核心)

**English.** A learned feature grid sampled by bilinear interpolation is a
powerful encoder, but naively growing it hits a **bottleneck**: adding grid
resolution/params yields diminishing PSNR (paper Fig. 5). PEPS breaks this by
sampling one shared grid at many Lissajous points. This notebook reproduces the
parameter-vs-PSNR curves for grid vs Grid-PEPS.

**繁體中文.** 用雙線性內插取樣的可學習特徵 grid 很強,但單純把它變大會遇到
**瓶頸**:增加解析度/參數,PSNR 收益遞減(論文 Fig.5)。PEPS 透過在多個
Lissajous 點取樣**同一個共享 grid** 打破瓶頸。本 notebook 重現 grid 與
Grid-PEPS 的「參數 vs PSNR」曲線。

In [1]:
# Repo bootstrap: make `peps` and `apps` importable from the notebook.
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import torch
from peps.train import auto_device
device = auto_device()
print('torch', torch.__version__, '| device', device)

torch 2.10.0+rocm7.0 | device cuda


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


## 1. Setup: one Kodak image / 設定:一張 Kodak 影像

In [2]:
from apps.image.data import load_image, image_to_coords_targets, find_kodak
img = load_image(find_kodak(1), max_size=256)
coords, targets, (H, W) = image_to_coords_targets(img)
print('image', H, W)

image 171 256


## 2. Sweep parameters: grid vs Grid-PEPS / 掃參數:grid vs Grid-PEPS
Same feature grid, but PEPS samples it at 2L+1 Lissajous points. 相同 grid,
但 PEPS 在 2L+1 個 Lissajous 點取樣。

In [3]:
from apps.image.build import build_grid, build_grid_peps
from peps.train import fit, TrainConfig, render_full
from peps.metrics import psnr

def run(model, pc, steps=1200):
    fit(model, coords, targets,
        TrainConfig(steps=steps, batch_size=16384, lr=1e-2, device=device))
    pred = render_full(model, coords, device=device).reshape(H, W, 3).clamp(0, 1)
    return pc, psnr(pred, img)

grid_curve, peps_curve = [], []
for res in [32, 48, 64, 96, 128]:
    grid_curve.append(run(*build_grid(resolution=res, feature_dim=4)))
    peps_curve.append(run(*build_grid_peps(resolution=res, feature_dim=4, num_frequencies=6)))
    print('res', res, '| grid', grid_curve[-1], '| peps', peps_curve[-1])

res 32 | grid (8771, 25.96417034974256) | peps (11843, 27.38591315251973)


res 48 | grid (13891, 27.804978314265686) | peps (16963, 28.958081841483367)


res 64 | grid (21059, 30.114179078942986) | peps (24131, 30.67045988810316)


res 96 | grid (41539, 34.858729777006644) | peps (44611, 36.05822967127032)


res 128 | grid (70211, 39.645028008798455) | peps (73283, 39.22776335818339)


## 3. Plot the bottleneck (reproduce Fig. 5) / 畫出瓶頸(重現 Fig.5)

In [4]:
import matplotlib.pyplot as plt
gx, gy = zip(*grid_curve); px, py = zip(*peps_curve)
plt.figure(figsize=(6, 4))
plt.plot(gx, gy, 'o-', label='grid baseline')
plt.plot(px, py, 's-', label='Grid-PEPS')
plt.xlabel('parameters'); plt.ylabel('PSNR (dB)')
plt.title('Fig.5-style: params vs PSNR'); plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

## 4. Takeaway / 小結
The grid baseline saturates; Grid-PEPS keeps climbing for the same parameter
budget. This is the motivational centerpiece of the whole course. Next: build
the PEPS wrapper formally (W04) and reproduce Table 1 on Kodak (W05).

grid baseline 會飽和;相同參數預算下 Grid-PEPS 持續上升。這是整門課的動機核心。
接著:正式建立 PEPS wrapper(W04),並在 Kodak 上重現 Table 1(W05)。